# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cassandragoto10-maker/Flyrank-ML-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

###SECTION 1 Baseline rule

I will prioritise pages where recent impressions have fallen substantially. Pages that are also at least 180 days since their last update receive additional priority because they may warrant a content refresh review.

The rule uses two observable signals: recent impressions change and days since last update. It does not use `trend_direction`, `trend_pct`, or any future-window information.

### Score

- Recent impressions decline greater than 50% = +3
- Recent impressions decline between 10% and 50% = +1
- At least 180 days since last update AND a recent impressions decline = +2 additional
- No qualifying signal = 0

### Reason codes

- `stale_and_strong_decline` — page is at least 180 days old since its last update and recent impressions declined by more than 50%.
- `stale_and_declining` — page is at least 180 days old since its last update and recent impressions declined by 10–50%.
- `strong_recent_decline` — recent impressions declined by more than 50%.
- `recent_decline` — recent impressions declined by 10–50%.
- `no_action` — neither decline condition is met.

### Action labels

- `review` — page has a positive baseline score and should be considered for review.
- `monitor` — page does not receive priority from this baseline.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
from google.colab import files
import os
import shutil

uploaded = files.upload()

filename = "content_refresh_anonymized.csv"

if filename not in uploaded:
    raise FileNotFoundError(
        f"Please upload {filename}"
    )

os.makedirs("/content/data/raw", exist_ok=True)

shutil.move(
    f"/content/{filename}",
    f"/content/data/raw/{filename}"
)

print("Dataset uploaded successfully.")
print("Location:")
print("/content/data/raw/content_refresh_anonymized.csv")

Saving content_refresh_anonymized.csv to content_refresh_anonymized.csv
Dataset uploaded successfully.
Location:
/content/data/raw/content_refresh_anonymized.csv


In [4]:
from pathlib import Path

DATA_PATH = Path("/content/data/raw/content_refresh_anonymized.csv")

print("Exists:", DATA_PATH.exists())
print("Size:", DATA_PATH.stat().st_size / 1024 / 1024, "MB")

Exists: True
Size: 6.416006088256836 MB


In [6]:
# ML-07 — Load the FlyRank starter dataset

import pandas as pd
from pathlib import Path

DATA_PATH = Path("/content/data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(DATA_PATH)

print(f"Loaded {len(df):,} rows")
print(f"Columns: {len(df.columns)}")
print("\nFirst 5 rows:")
display(df.head())

Loaded 30,000 rows
Columns: 44

First 5 rows:


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [7]:
# Check the dataset structure

print("Dataset shape:", df.shape)

print("\nColumns:")
for i, column in enumerate(df.columns, start=1):
    print(f"{i}. {column}")

Dataset shape: (30000, 44)

Columns:
1. content_id
2. client_id
3. search_volume
4. competition
5. competition_level
6. cpc
7. content_type
8. main_intent
9. word_count
10. char_count
11. provider_used
12. model_used
13. impressions_90d
14. clicks_90d
15. pageviews_90d
16. sessions_90d
17. users_90d
18. engaged_sessions_90d
19. ai_sessions_90d
20. scroll_events_90d
21. days_with_impressions
22. days_with_sessions
23. impressions_last_30d
24. clicks_last_30d
25. sessions_last_30d
26. impressions_prev_30d
27. clicks_prev_30d
28. sessions_prev_30d
29. content_age_days
30. age_tier
31. age_tier_order
32. days_since_last_update
33. freshness_tier
34. word_count_tier
35. char_count_tier
36. ctr
37. avg_position
38. engagement_rate
39. scroll_rate
40. ai_traffic_pct
41. impression_tier
42. position_tier
43. trend_direction
44. trend_pct


In [8]:
# ML-07 Section 2 — Signal check 1: staleness

import numpy as np
import pandas as pd

# Create staleness buckets
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-np.inf, 89, 179, 269, np.inf],
    labels=["<90 days", "90–179 days", "180–269 days", "270+ days"]
)

staleness_check = (
    df.groupby("staleness_bucket", observed=False)
      .size()
      .reset_index(name="n")
)

print("Signal 1: Days since last update")
display(staleness_check)

Signal 1: Days since last update


,staleness_bucket,n
0,<90 days,20655
1,90–179 days,9171
2,180–269 days,139
3,270+ days,35


In [9]:
# ML-07 Section 2 — Signal check 2: recent impressions change

# Calculate percentage change from the previous 30 days.
# When the previous period has zero impressions, the percentage change
# is not treated as a meaningful decline.
df["recent_impression_change_pct"] = np.where(
    df["impressions_prev_30d"] > 0,
    (
        (df["impressions_last_30d"] - df["impressions_prev_30d"])
        / df["impressions_prev_30d"]
    ) * 100,
    np.nan
)

df["recent_change_bucket"] = pd.cut(
    df["recent_impression_change_pct"],
    bins=[-np.inf, -50, -10, 10, 50, np.inf],
    labels=[
        "< -50%",
        "-50% to -10%",
        "-10% to +10%",
        "+10% to +50%",
        "> +50%"
    ]
)

impression_change_check = (
    df.groupby("recent_change_bucket", observed=False)
      .size()
      .reset_index(name="n")
)

print("Signal 2: Recent impressions change")
display(impression_change_check)

Signal 2: Recent impressions change


,recent_change_bucket,n
0,< -50%,9642
1,-50% to -10%,8626
2,-10% to +10%,3016
3,+10% to +50%,2713
4,> +50%,2615


In [11]:
# ML-07 — Signal verdicts
# The 30,000-row starter CSV does not contain is_declining_label.
# Therefore, no label-based validation is performed here.

print("=== Signal 1: Days since last update ===")

print("Bucket counts:")
display(staleness_check)

print("\nSignal 1 verdict: MIXED")

print(
    "Reason: Most observations are below 180 days since the last update, "
    "while only a small number are in the 180+ day buckets. "
    "The signal is directly related to content freshness, but its high-staleness "
    "threshold has limited coverage in this dataset."
)


print("\n=== Signal 2: Recent impressions change ===")

print("Bucket counts:")
display(impression_change_check)

print("\nSignal 2 verdict: CONFIRMED")

print(
    "Reason: Recent impressions change provides substantial coverage across "
    "the dataset, including a large group with declines greater than 50% and "
    "another large group with declines between 50% and 10%. "
    "It is therefore a usable observable signal for prioritising pages for review."
)


print("\n=== Label availability check ===")

if "is_declining_label" in df.columns:
    print("is_declining_label is available.")
else:
    print(
        "is_declining_label is NOT present in the starter CSV. "
        "No label-based precision or decline-rate calculation is performed "
        "from this dataset."
    )

=== Signal 1: Days since last update ===
Bucket counts:


,staleness_bucket,n
0,<90 days,20655
1,90–179 days,9171
2,180–269 days,139
3,270+ days,35



Signal 1 verdict: MIXED
Reason: Most observations are below 180 days since the last update, while only a small number are in the 180+ day buckets. The signal is directly related to content freshness, but its high-staleness threshold has limited coverage in this dataset.

=== Signal 2: Recent impressions change ===
Bucket counts:


,recent_change_bucket,n
0,< -50%,9642
1,-50% to -10%,8626
2,-10% to +10%,3016
3,+10% to +50%,2713
4,> +50%,2615



Signal 2 verdict: CONFIRMED
Reason: Recent impressions change provides substantial coverage across the dataset, including a large group with declines greater than 50% and another large group with declines between 50% and 10%. It is therefore a usable observable signal for prioritising pages for review.

=== Label availability check ===
is_declining_label is NOT present in the starter CSV. No label-based precision or decline-rate calculation is performed from this dataset.


SECTION 2

In [12]:
# ML-07 Section 2 — Build the transparent ranked baseline

import os
import numpy as np
import pandas as pd

# -----------------------------
# 1. Define transparent signals
# -----------------------------

recent_decline = df["recent_impression_change_pct"] < -10
strong_recent_decline = df["recent_impression_change_pct"] < -50
stale = df["days_since_last_update"] >= 180

# -----------------------------
# 2. Calculate transparent score
# -----------------------------

df["baseline_score"] = (
    strong_recent_decline.astype(int) * 3
    + recent_decline.astype(int) * 1
    + (stale & recent_decline).astype(int) * 2
)

# -----------------------------
# 3. Assign ONE reason code
# -----------------------------

df["reason_code"] = np.select(
    [
        stale & strong_recent_decline,
        stale & recent_decline,
        strong_recent_decline,
        recent_decline
    ],
    [
        "stale_and_strong_decline",
        "stale_and_declining",
        "strong_recent_decline",
        "recent_decline"
    ],
    default="no_action"
)

# -----------------------------
# 4. Assign action
# -----------------------------

df["action"] = np.where(
    df["baseline_score"] > 0,
    "review",
    "monitor"
)

# -----------------------------
# 5. Rank all observations
# -----------------------------

df["rank"] = (
    df["baseline_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

# -----------------------------
# 6. Create ranked queue
# -----------------------------

baseline_queue = (
    df[
        [
            "content_id",
            "client_id",
            "baseline_score",
            "reason_code",
            "action",
            "rank"
        ]
    ]
    .sort_values(
        ["baseline_score", "rank"],
        ascending=[False, True]
    )
)

# -----------------------------
# 7. Write required CSV
# -----------------------------

os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"

baseline_queue.to_csv(
    output_path,
    index=False
)

print("Baseline queue created successfully.")
print(f"Rows: {len(baseline_queue):,}")
print(f"Output: {output_path}")

print("\nScore distribution:")
print(df["baseline_score"].value_counts().sort_index())

print("\nTop 20:")
display(baseline_queue.head(20))

Baseline queue created successfully.
Rows: 30,000
Output: work/outputs/baseline_action_score.csv

Score distribution:
baseline_score
0    11741
1     8860
3       30
4     9311
6       58
Name: count, dtype: int64

Top 20:


,content_id,client_id,baseline_score,reason_code,action,rank
91,content_48724397d104,client_d4735e3a26,6,stale_and_strong_decline,review,1
698,content_b16bd7307b39,client_7f2253d7e2,6,stale_and_strong_decline,review,2
1227,content_4f241bad48a3,client_6208ef0f77,6,stale_and_strong_decline,review,3
1659,content_bbca724138f2,client_6208ef0f77,6,stale_and_strong_decline,review,4
3280,content_a34d943a132c,client_d029fa3a95,6,stale_and_strong_decline,review,5
4754,content_164eee6bf9c1,client_d029fa3a95,6,stale_and_strong_decline,review,6
5327,content_fe16a55cd13d,client_7f2253d7e2,6,stale_and_strong_decline,review,7
5653,content_10b9f5f766b4,client_d4735e3a26,6,stale_and_strong_decline,review,8
6119,content_cd27391ecd03,client_d029fa3a95,6,stale_and_strong_decline,review,9
6421,content_fc8cb7532683,client_d029fa3a95,6,stale_and_strong_decline,review,10


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 review

The baseline ranks content for review when it is both stale and showing a strong recent decline in impressions. The highest-scoring items therefore receive the `review` action and the `stale_and_strong_decline` reason code.

The confidence note reflects how directly the observed signals support the rule. A high-confidence pick has clear staleness and a substantial decline. A lower-confidence pick may have a strong percentage decline but very low absolute impressions, making the percentage change less informative.

For each item, I also record what could make the recommendation wrong. This is important because a decline in impressions does not by itself prove that updating the content will improve performance.

In [13]:
# ML-07 Section 3 — Prepare the Top-20 review

top20 = (
    df[
        [
            "content_id",
            "client_id",
            "baseline_score",
            "reason_code",
            "action",
            "rank",
            "days_since_last_update",
            "impressions_last_30d",
            "impressions_prev_30d",
            "recent_impression_change_pct"
        ]
    ]
    .sort_values("rank")
    .head(20)
    .copy()
)

# Add a simple confidence note based only on the strength of
# the signals used by the rule.
top20["confidence_note"] = np.select(
    [
        top20["baseline_score"] == 6,
        top20["baseline_score"] == 4,
        top20["baseline_score"] == 3,
        top20["baseline_score"] == 1
    ],
    [
        "Higher confidence: both staleness and a strong recent impressions decline support review.",
        "Moderate-high confidence: staleness and a recent impressions decline support review.",
        "Moderate confidence: a strong recent impressions decline supports review.",
        "Lower confidence: the page meets the weaker recent-decline condition only."
    ],
    default="Low confidence: no positive baseline signal."
)

# Add what could make the rule wrong.
top20["what_would_make_it_wrong"] = np.select(
    [
        top20["baseline_score"] == 6,
        top20["baseline_score"] == 4,
        top20["baseline_score"] == 3,
        top20["baseline_score"] == 1
    ],
    [
        "The impressions decline could reflect temporary demand or measurement variation, and the page may not benefit from a refresh.",
        "The recent decline may be temporary, and age alone does not prove that updating the page will improve performance.",
        "A large impressions decline may be caused by external search-demand changes rather than content quality.",
        "A smaller impressions decline may be normal variation and may not justify a refresh."
    ],
    default="The rule may have missed an important factor not included in the baseline."
)

display(top20)

,content_id,client_id,baseline_score,reason_code,action,rank,days_since_last_update,impressions_last_30d,impressions_prev_30d,recent_impression_change_pct,confidence_note,what_would_make_it_wrong
91,content_48724397d104,client_d4735e3a26,6,stale_and_strong_decline,review,1,211,4,12,-66.666667,Higher confidence: both staleness and a strong...,The impressions decline could reflect temporar...
698,content_b16bd7307b39,client_7f2253d7e2,6,stale_and_strong_decline,review,2,194,554,1831,-69.743310,Higher confidence: both staleness and a strong...,The impressions decline could reflect temporar...
1227,content_4f241bad48a3,client_6208ef0f77,6,stale_and_strong_decline,review,3,236,26,187,-86.096257,Higher confidence: both staleness and a strong...,The impressions decline could reflect temporar...
1659,content_bbca724138f2,client_6208ef0f77,6,stale_and_strong_decline,review,4,236,0,52,-100.000000,Higher confidence: both staleness and a strong...,The impressions decline could reflect temporar...
3280,content_a34d943a132c,client_d029fa3a95,6,stale_and_strong_decline,review,5,183,10,21,-52.380952,Higher confidence: both staleness and a strong...,The impressions decline could reflect temporar...
4754,content_164eee6bf9c1,client_d029fa3a95,6,stale_and_strong_decline,review,6,183,0,7,-100.000000,Higher confidence: both staleness and a strong...,The impressions decline could reflect temporar...
5327,content_fe16a55cd13d,client_7f2253d7e2,6,stale_and_strong_decline,review,7,194,746,1562,-52.240717,Higher confidence: both staleness and a strong...,The impressions decline could reflect temporar...
5653,content_10b9f5f766b4,client_d4735e3a26,6,stale_and_strong_decline,review,8,211,1,9,-88.888889,Higher confidence: both staleness and a strong...,The impressions decline could reflect temporar...
6119,content_cd27391ecd03,client_d029fa3a95,6,stale_and_strong_decline,review,9,183,6,15,-60.000000,Higher confidence: both staleness and a strong...,The impressions decline could reflect temporar...
6421,content_fc8cb7532683,client_d029fa3a95,6,stale_and_strong_decline,review,10,183,3,8,-62.500000,Higher confidence: both staleness and a strong...,The impressions decline could reflect temporar...


In [14]:
# Inspect the actual signal strength for the tied top-20

display(
    top20[
        [
            "rank",
            "content_id",
            "baseline_score",
            "reason_code",
            "days_since_last_update",
            "impressions_last_30d",
            "impressions_prev_30d",
            "recent_impression_change_pct",
            "action"
        ]
    ]
)

,rank,content_id,baseline_score,reason_code,days_since_last_update,impressions_last_30d,impressions_prev_30d,recent_impression_change_pct,action
91,1,content_48724397d104,6,stale_and_strong_decline,211,4,12,-66.666667,review
698,2,content_b16bd7307b39,6,stale_and_strong_decline,194,554,1831,-69.743310,review
1227,3,content_4f241bad48a3,6,stale_and_strong_decline,236,26,187,-86.096257,review
1659,4,content_bbca724138f2,6,stale_and_strong_decline,236,0,52,-100.000000,review
3280,5,content_a34d943a132c,6,stale_and_strong_decline,183,10,21,-52.380952,review
4754,6,content_164eee6bf9c1,6,stale_and_strong_decline,183,0,7,-100.000000,review
5327,7,content_fe16a55cd13d,6,stale_and_strong_decline,194,746,1562,-52.240717,review
5653,8,content_10b9f5f766b4,6,stale_and_strong_decline,211,1,9,-88.888889,review
6119,9,content_cd27391ecd03,6,stale_and_strong_decline,183,6,15,-60.000000,review
6421,10,content_fc8cb7532683,6,stale_and_strong_decline,183,3,8,-62.500000,review


In [16]:
# ML-07 Section 3 — Top-20 review

# Get the top 20 ranks from the baseline queue
top20_ids = baseline_queue.head(20)["content_id"]

# Pull the original signal columns from df
top20_review = df[df["content_id"].isin(top20_ids)].copy()

# Keep the original ranking information from the baseline queue
rank_info = baseline_queue[
    [
        "content_id",
        "baseline_score",
        "reason_code",
        "action",
        "rank"
    ]
].copy()

# Merge the ranking information with the original signal data
top20_review = rank_info.merge(
    top20_review[
        [
            "content_id",
            "days_since_last_update",
            "impressions_last_30d",
            "impressions_prev_30d",
            "recent_impression_change_pct"
        ]
    ],
    on="content_id",
    how="left"
)

# Add a confidence note
def confidence_note(row):

    if (
        row["days_since_last_update"] >= 180
        and row["recent_impression_change_pct"] <= -50
        and row["impressions_prev_30d"] >= 100
    ):
        return "High: stale and strong decline with meaningful prior volume"

    elif (
        row["days_since_last_update"] >= 180
        and row["recent_impression_change_pct"] <= -50
    ):
        return "Moderate: stale and strong decline, but prior volume is low"

    else:
        return "Lower: weaker evidence than the highest-ranked cases"


top20_review["confidence_note"] = top20_review.apply(
    confidence_note,
    axis=1
)

# Add the required skeptic's-eye review
top20_review["what_would_make_it_wrong"] = (
    "The decline may be temporary, caused by seasonality, "
    "search-demand changes, or another factor unrelated to content freshness."
)

# Sort by rank
top20_review = top20_review.sort_values("rank")

# Display the completed review
display(
    top20_review[
        [
            "rank",
            "content_id",
            "baseline_score",
            "reason_code",
            "action",
            "days_since_last_update",
            "impressions_last_30d",
            "impressions_prev_30d",
            "recent_impression_change_pct",
            "confidence_note",
            "what_would_make_it_wrong"
        ]
    ]
)

,rank,content_id,baseline_score,reason_code,action,days_since_last_update,impressions_last_30d,impressions_prev_30d,recent_impression_change_pct,confidence_note,what_would_make_it_wrong
0,1,content_48724397d104,6,stale_and_strong_decline,review,211.0,4.0,12.0,-66.666667,"Moderate: stale and strong decline, but prior ...","The decline may be temporary, caused by season..."
1,2,content_b16bd7307b39,6,stale_and_strong_decline,review,194.0,554.0,1831.0,-69.743310,High: stale and strong decline with meaningful...,"The decline may be temporary, caused by season..."
2,3,content_4f241bad48a3,6,stale_and_strong_decline,review,236.0,26.0,187.0,-86.096257,High: stale and strong decline with meaningful...,"The decline may be temporary, caused by season..."
3,4,content_bbca724138f2,6,stale_and_strong_decline,review,236.0,0.0,52.0,-100.000000,"Moderate: stale and strong decline, but prior ...","The decline may be temporary, caused by season..."
4,5,content_a34d943a132c,6,stale_and_strong_decline,review,183.0,10.0,21.0,-52.380952,"Moderate: stale and strong decline, but prior ...","The decline may be temporary, caused by season..."
...,...,...,...,...,...,...,...,...,...,...,...
29995,29996,content_fcad8c75dc44,0,no_action,monitor,NaN,NaN,NaN,NaN,Lower: weaker evidence than the highest-ranked...,"The decline may be temporary, caused by season..."
29996,29997,content_23dce6a656e6,0,no_action,monitor,NaN,NaN,NaN,NaN,Lower: weaker evidence than the highest-ranked...,"The decline may be temporary, caused by season..."
29997,29998,content_9bd30342fd4a,0,no_action,monitor,NaN,NaN,NaN,NaN,Lower: weaker evidence than the highest-ranked...,"The decline may be temporary, caused by season..."
29998,29999,content_c322796023c8,0,no_action,monitor,NaN,NaN,NaN,NaN,Lower: weaker evidence than the highest-ranked...,"The decline may be temporary, caused by season..."


### Top-20 review

The baseline ranks pages for review using a transparent rule based on recent impressions change and content freshness. The top 20 all received the maximum baseline score of 6 because they meet both conditions: they are at least 180 days since their last update and their recent impressions declined by more than 50%.

The tied scores should not be interpreted as a meaningful ranking among these 20 pages. The rule identifies them as a priority group rather than proving that rank 1 is more important than rank 20.

Each row is reviewed using its action, reason code, the strength of the observed signals, and a condition that could make the recommendation wrong.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

# ML-07 Section 3 — Top-20 review

# Get the top 20 ranks from the baseline queue
top20_ids = baseline_queue.head(20)["content_id"]

# Pull the original signal columns from df
top20_review = df[df["content_id"].isin(top20_ids)].copy()

# Keep the original ranking information from the baseline queue
rank_info = baseline_queue[
    [
        "content_id",
        "baseline_score",
        "reason_code",
        "action",
        "rank"
    ]
].copy()

# Merge the ranking information with the original signal data
top20_review = rank_info.merge(
    top20_review[
        [
            "content_id",
            "days_since_last_update",
            "impressions_last_30d",
            "impressions_prev_30d",
            "recent_impression_change_pct"
        ]
    ],
    on="content_id",
    how="left"
)

# Add a confidence note
def confidence_note(row):

    if (
        row["days_since_last_update"] >= 180
        and row["recent_impression_change_pct"] <= -50
        and row["impressions_prev_30d"] >= 100
    ):
        return "High: stale and strong decline with meaningful prior volume"

    elif (
        row["days_since_last_update"] >= 180
        and row["recent_impression_change_pct"] <= -50
    ):
        return "Moderate: stale and strong decline, but prior volume is low"

    else:
        return "Lower: weaker evidence than the highest-ranked cases"


top20_review["confidence_note"] = top20_review.apply(
    confidence_note,
    axis=1
)

# Add the required skeptic's-eye review
top20_review["what_would_make_it_wrong"] = (
    "The decline may be temporary, caused by seasonality, "
    "search-demand changes, or another factor unrelated to content freshness."
)

# Sort by rank
top20_review = top20_review.sort_values("rank")

# Display the completed review
display(
    top20_review[
        [
            "rank",
            "content_id",
            "baseline_score",
            "reason_code",
            "action",
            "days_since_last_update",
            "impressions_last_30d",
            "impressions_prev_30d",
            "recent_impression_change_pct",
            "confidence_note",
            "what_would_make_it_wrong"
        ]
    ]
)

In [17]:
# ML-07 Section 4 — Weak picks + leakage check

print("=== Weak picks ===")

# Find high-scoring items where the previous-period impression volume is low.
# These are useful examples of where a percentage decline may look stronger
# than the underlying absolute change.
weak_picks = baseline_queue[
    (baseline_queue["baseline_score"] >= 6)
].copy()

# Bring the original signal columns back from df
weak_picks = weak_picks.merge(
    df[
        [
            "content_id",
            "days_since_last_update",
            "impressions_last_30d",
            "impressions_prev_30d",
            "recent_impression_change_pct"
        ]
    ],
    on="content_id",
    how="left"
)

# Focus on high-score picks with low previous-period volume
weak_picks_low_volume = weak_picks[
    weak_picks["impressions_prev_30d"] < 100
].copy()

print(
    "Number of high-score picks with previous-period impressions < 100:",
    len(weak_picks_low_volume)
)

print("\nExamples of potentially weak picks:")

display(
    weak_picks_low_volume[
        [
            "rank",
            "content_id",
            "baseline_score",
            "reason_code",
            "days_since_last_update",
            "impressions_last_30d",
            "impressions_prev_30d",
            "recent_impression_change_pct",
            "action"
        ]
    ]
    .sort_values("rank")
    .head(10)
)


# ============================================================
# Leakage check
# ============================================================

print("\n=== Leakage check ===")

# These fields are not allowed as baseline inputs because they are
# derived from the decline outcome or are the label itself.
forbidden_features = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

print("Label-derived / prohibited fields:")

for feature in forbidden_features:
    if feature in df.columns:
        print(f"- {feature}: PRESENT in dataset, NOT USED in baseline score")
    else:
        print(f"- {feature}: NOT PRESENT in dataset")


# Confirm the actual variables used to calculate the score
print("\nActual baseline signal inputs:")
print("- days_since_last_update")
print("- recent_impression_change_pct")

print("\nFuture-window check:")
print(
    "The rule uses impressions_last_30d and impressions_prev_30d "
    "from the supplied historical feature window."
)

print(
    "No future outcome, trend_direction, trend_pct, or "
    "decline label was used to calculate baseline_score."
)

print("\nLeakage check completed.")

=== Weak picks ===
Number of high-score picks with previous-period impressions < 100: 45

Examples of potentially weak picks:


,rank,content_id,baseline_score,reason_code,days_since_last_update,impressions_last_30d,impressions_prev_30d,recent_impression_change_pct,action
0,1,content_48724397d104,6,stale_and_strong_decline,211,4,12,-66.666667,review
3,4,content_bbca724138f2,6,stale_and_strong_decline,236,0,52,-100.000000,review
4,5,content_a34d943a132c,6,stale_and_strong_decline,183,10,21,-52.380952,review
5,6,content_164eee6bf9c1,6,stale_and_strong_decline,183,0,7,-100.000000,review
7,8,content_10b9f5f766b4,6,stale_and_strong_decline,211,1,9,-88.888889,review
8,9,content_cd27391ecd03,6,stale_and_strong_decline,183,6,15,-60.000000,review
9,10,content_fc8cb7532683,6,stale_and_strong_decline,183,3,8,-62.500000,review
10,11,content_9b7a283bd1c9,6,stale_and_strong_decline,183,10,22,-54.545455,review
11,12,content_f01216059a6a,6,stale_and_strong_decline,335,9,31,-70.967742,review
13,14,content_8f2c815af658,6,stale_and_strong_decline,301,1,3,-66.666667,review



=== Leakage check ===
Label-derived / prohibited fields:
- trend_direction: PRESENT in dataset, NOT USED in baseline score
- trend_pct: PRESENT in dataset, NOT USED in baseline score
- is_declining_label: NOT PRESENT in dataset

Actual baseline signal inputs:
- days_since_last_update
- recent_impression_change_pct

Future-window check:
The rule uses impressions_last_30d and impressions_prev_30d from the supplied historical feature window.
No future outcome, trend_direction, trend_pct, or decline label was used to calculate baseline_score.

Leakage check completed.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.